In [ ]:
# Load libraries
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from econml.dml import LinearDML, SparseLinearDML, CausalForestDML, NonParamDML
from econml.grf import CausalForest
# To compute RMSE
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error

In [2]:
# Load the experiment dataset
datos = pd.read_csv('/Users/carloseqa/Github/Datasets/escenarioA1.csv')
nuevos = pd.read_csv('/Users/carloseqa/Github/Datasets/escenarioA2.csv')
# Solucion
solucionA = pd.read_csv('/Users/carloseqa/Desktop/trabajo6/solucionA.csv')

In [4]:
# Definicion de variables para datos de los que aprenderemos los modelos
y = datos['Y']
T = datos['T']
X = datos.drop(['id','Y','T'],axis=1)
X2 = nuevos.drop(['id'],axis=1)

In [5]:
# Modelos para g(X,W) y m(X,W)
model_g = LinearRegression()
model_m = DummyClassifier(strategy='prior')
# Modelo 01 : Linear DML
est1 = LinearDML(model_y=model_g, model_t=model_m,discrete_treatment=True)
est1.fit(y,T,X=X)
efecto1 = est1.effect(X2)
decision1 = np.where(efecto1 > 0,1,0)

In [6]:
# Modelos para g(X,W) y m(X,W)
model_g = LinearRegression()
model_m = DummyClassifier(strategy='prior')
# Modelo 02 : Sparse Linear DML
est2 = SparseLinearDML(model_y=model_g, model_t=model_m,discrete_treatment=True)
est2.fit(y,T,X=X)
efecto2 = est2.effect(X2)
decision2 = np.where(efecto2 > 0,1,0)

In [7]:
# Modelos para g(X,W) y m(X,W)
model_g = LinearRegression()
model_m = DummyClassifier(strategy='prior')
# Modelo 03 : CausalForestDML
est3 = CausalForestDML(model_t=model_m,model_y=model_g,discrete_treatment=True)  # ,
est3.fit(y,T,X=X)
efecto3 = est3.effect(X2)
decision3 = np.where(efecto3 > 0,1,0)

In [8]:
# Modelo 04 : Generalized Random Forest
est4 = CausalForest(random_state=1234)
est4.fit(X,T,y)
efecto4 = est4.predict(X2).ravel()
decision4 = np.where(efecto4 > 0,1,0)

In [ ]:
rmse1 = root_mean_squared_error(solucion['tau'],efecto1)
rmse2 = root_mean_squared_error(solucion['tau'],efecto2)
rmse3 = root_mean_squared_error(solucion['tau'],efecto3)
rmse4 = root_mean_squared_error(solucion['tau'],efecto4)

print('RMSE Modelo LinearDML: ',rmse1)
print('RMSE Modelo SparseLinearDML: ',rmse2)
print('RMSE Modelo CausalForestDML: ',rmse3)
print('RMSE Modelo grf.CausalForest: ',rmse4)
print('')

mape1 = mean_absolute_percentage_error(solucion['tau'],efecto1)
mape2 = mean_absolute_percentage_error(solucion['tau'],efecto2)
mape3 = mean_absolute_percentage_error(solucion['tau'],efecto3)
mape4 = mean_absolute_percentage_error(solucion['tau'],efecto4)

print('MAPE Modelo LinearDML: ',mape1)
print('MAPE Modelo SparseLinearDML: ',mape2)
print('MAPE Modelo CausalForestDML: ',mape3)
print('MAPE Modelo grf.CausalForest: ',mape4)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(10, 8))  # 2x2 grid of subplots

# First plot
sns.scatterplot(x=nuevos['X1'], y=efecto1, ax=axes[0, 0])
axes[0, 0].set_title("Effect 1")

# Second plot
sns.scatterplot(x=nuevos['X1'], y=efecto2, ax=axes[0, 1])
axes[0, 1].set_title("Effect 2")

# Third plot
sns.scatterplot(x=nuevos['X1'], y=efecto3, ax=axes[1, 0])
axes[1, 0].set_title("Effect 3")

# Fourth plot
sns.scatterplot(x=nuevos['X1'], y=efecto4, ax=axes[1, 1])
axes[1, 1].set_title("Effect 4")

# Adjust layout
plt.tight_layout()
plt.show()
